In [2]:
#code for adding flow particles to phi events. use p-pB hist to get the eta and pt, making hist if below somewhere

#Ai version of the above with histogram eta, pt, taken from p-Pb

# adding isotropic particles to p-p, memory-optimized (chunked/streaming) version
# -- eta/pt now sampled from PRE-SAVED histograms instead of bootstrapping
#    from the dataset itself, so this is now a single streaming pass.
#
# Key memory features:
#   1. Never loads the whole dataset into memory at once -- the tree is
#      streamed in chunks via tree.iterate(...).
#   2. Never converts branches into Python lists of numpy arrays (that
#      per-event object overhead is enormous at millions of events) --
#      everything stays as awkward arrays / plain numpy arrays.
#   3. sample_phi_exact and sample_from_histogram are fully vectorized
#      with numpy (no per-sample Python-level loop).
#   4. Generated arrays use float32 (matching typical ntuple branch dtype)
#      instead of float64, halving the memory of the new-particle arrays,
#      which are the single biggest memory consumer (n_events * 110).
#   5. Per-chunk temporaries are deleted and gc.collect()'d so peak memory
#      stays roughly constant regardless of total dataset size.

import gc
import numpy as np
import awkward as ak
import uproot as ur

#IN_PATH = r"data/pp_mb_fixed.root" 
IN_PATH = r"data/2merged_100_200.root"
OUT_PATH = r"data/pPb_isomerge_v205_10000Nparticles.root" 
#OUT_PATH = r"data/pPb_isomerge_v205_50Nparticles.root"
HIST_PATH = r"data/p_Pb_eta_pt_histograms.npz"  
 
startt = 0
endd = 100000
MIN_MULT = 10
N_NEW_PER_EVENT = 10000
v2= .05

# Tune this based on available RAM. At CHUNK_SIZE events per chunk, the
# generated-particle arrays are ~CHUNK_SIZE * N_NEW_PER_EVENT floats each
# (phi/eta/pt), so e.g. 200_000 * 110 * 4 bytes (float32) ~= 88 MB per
# branch per chunk -- easily bounded regardless of how many total events
# the file has.
CHUNK_SIZE = 50_000

rng = np.random.default_rng()

# ----------------------------------------------------------------------
# Vectorized exact-inversion sampler for f(phi) = (1 + 2*v*cos(2phi)) / (2*pi)
# v=0 gives a perfectly isotropic (uniform) distribution.
# ----------------------------------------------------------------------
def sample_phi_exact_vec(n, v, dtype=np.float32):
    u = rng.random(n)
    phi_ = 2 * np.pi * u - np.pi          # initial guess: uniform
    for _ in range(4):                     # Newton iterations
        F = (phi_ + np.pi + v * np.sin(2 * phi_)) / (2 * np.pi) - u
        dF = (1 + 2 * v * np.cos(2 * phi_)) / (2 * np.pi)
        phi_ -= F / dF
    return phi_.astype(dtype, copy=False)


# ----------------------------------------------------------------------
# Histogram-based sampler: pick a bin according to its probability
# (counts/sum(counts)), then draw uniformly within that bin's edges.
# Vectorized over n at once.
# ----------------------------------------------------------------------
def sample_from_histogram(counts, edges, n, dtype=np.float32):
    probs = counts / counts.sum()
    bin_idx = rng.choice(len(counts), size=n, p=probs)
    left = edges[bin_idx]
    right = edges[bin_idx + 1]
    return rng.uniform(left, right).astype(dtype, copy=False)


# ----------------------------------------------------------------------
# Load the pre-saved eta/pt histograms once (small, constant memory --
# no need to build a pool from the dataset itself anymore).
# ----------------------------------------------------------------------
hist_data = np.load(HIST_PATH)
eta_counts, eta_edges = hist_data["eta_counts"], hist_data["eta_edges"]
pt_counts, pt_edges = hist_data["pt_counts"], hist_data["pt_edges"]

n_events_survivors = 0 
N_total = 0 

# ----------------------------------------------------------------------
# SINGLE PASS: stream through the file, apply the multiplicity cut,
# generate the new particles per surviving event from the histograms
# above, merge them onto the end of each event's original arrays, and
# write the result straight to the output file chunk-by-chunk (no
# full-dataset array is ever held in memory).
# ----------------------------------------------------------------------
with ur.open(IN_PATH) as file, ur.recreate(OUT_PATH) as fout:
    tree = file["tree;1"]  # I have no clue why its jet_tree;1;1
    first_chunk = True

    for chunk in tree.iterate(
        expressions=["phi", "pt", "eta", "weight"],
        entry_start=startt,
        entry_stop=endd,
        step_size=CHUNK_SIZE,
        library="ak",
    ):
        lengths_chunk = ak.num(chunk["phi"], axis=1)
        mask = lengths_chunk >= MIN_MULT

        phi_masked = chunk["phi"][mask]
        eta_masked = chunk["eta"][mask]
        pt_masked = chunk["pt"][mask]
        weight_masked = chunk["weight"][mask]

        n_chunk_survivors = int(ak.sum(mask))
        total_new_chunk = n_chunk_survivors * N_NEW_PER_EVENT

        # Isotropic-ish phi for the new particles (vectorized, float32)
        new_phi_flat = sample_phi_exact_vec(total_new_chunk, v2, dtype=np.float32)
        new_phi_2d = new_phi_flat.reshape(n_chunk_survivors, N_NEW_PER_EVENT)

        # Sample eta/pt from the pre-saved histograms
        new_eta_2d = sample_from_histogram(eta_counts, eta_edges, total_new_chunk).reshape(
            n_chunk_survivors, N_NEW_PER_EVENT
        )
        new_pt_2d = sample_from_histogram(pt_counts, pt_edges, total_new_chunk).reshape(
            n_chunk_survivors, N_NEW_PER_EVENT
        )

        # Merge original (jagged) + new (regular) per event -> jagged
        final_phi_chunk = ak.concatenate([phi_masked, ak.Array(new_phi_2d)], axis=1)
        final_eta_chunk = ak.concatenate([eta_masked, ak.Array(new_eta_2d)], axis=1)
        final_pt_chunk = ak.concatenate([pt_masked, ak.Array(new_pt_2d)], axis=1)
        final_weight_chunk = weight_masked  # unchanged, not extended for new particles

        out_data = {
            "phi": final_phi_chunk,
            "eta": final_eta_chunk,
            "pt": final_pt_chunk,
            "weight": final_weight_chunk,
        }

        if first_chunk:
            fout["tree"] = out_data
            first_chunk = False
        else:
            fout["tree"].extend(out_data)

        n_events_survivors += n_chunk_survivors
        N_total += int(ak.sum(lengths_chunk[mask]))

        del (chunk, lengths_chunk, mask, phi_masked, eta_masked, pt_masked, weight_masked,
            new_phi_flat, new_phi_2d, new_eta_2d, new_pt_2d,
            final_phi_chunk, final_eta_chunk, final_pt_chunk, final_weight_chunk, out_data)
        gc.collect()

print(f"Wrote {OUT_PATH}") 
print(f"Events: {n_events_survivors}, original particles: {N_total}, "
      f"new particles per event: {N_NEW_PER_EVENT}") 

Wrote data/pPb_isomerge_v205_10000Nparticles.root
Events: 100000, original particles: 16045379, new particles per event: 10000


In [1]:
#merging files to have mults that I want
import awkward as ak 
import uproot

input_files = [
    "data/last_Hijing_1M.root",
    "data/last_Hijing_4M.root",
    "data/last_Hijing_10M.root", 
    "data/Hijing_p_Pb10M.root",
    "data/Hijing_p_Pb10M3.root",
]

output_file = "data/2merged_100_200.root"

branches = ["phi", "pt", "eta", "weight"]

with uproot.recreate(output_file) as fout:

    first = True

    for filename in input_files:
        print(filename)

        for arrays in uproot.iterate(
            f"{filename}:tree",
            branches,
            step_size="100 MB",
            library="ak",
        ):

            mult = ak.sum(arrays["pt"] > 0.4, axis=1)
            mask = (mult >= 100) & (mult <= 200)

            if not ak.any(mask):
                continue

            selected = {b: arrays[b][mask] for b in branches}

            if first:
                fout["tree"] = selected
                first = False
            else:
                fout["tree"].extend(selected)

print("Done.")

data/last_Hijing_1M.root
data/last_Hijing_4M.root
data/last_Hijing_10M.root
data/Hijing_p_Pb10M.root
data/Hijing_p_Pb10M3.root
Done.


In [1]:
#for making Et cuts

##### Et cuts
import awkward as ak 
import uproot

input_files = [
    "data/Hijing_Et_testing.root" 
]

output_file = "data/Et_21-60.root"

branches = ["phi", "pt", "eta", "weight", "sumEt"]

with uproot.recreate(output_file) as fout:

    first = True

    for filename in input_files:
        print(filename)
        #print(file.keys())

        for arrays in uproot.iterate(
            f"{filename}:tree",
            #f"{filename}:tree;1",
            branches,
            step_size="100 MB",
            library="ak",
        ):

            #mult = ak.sum(arrays["pt"] > 0.4, axis=1)
            mask = (arrays["sumEt"] >= 21) & (arrays["sumEt"] <= 60)

            if not ak.any(mask):
                continue

            selected = {b: arrays[b][mask] for b in branches}

            if first:
                fout["tree"] = selected
                first = False
            else: 
                fout["tree"].extend(selected)

print("Done.") 

data/Hijing_Et_testing.root
Done.


In [1]:
# adding isometric data shitt to p-Pb, set amount of points

import numpy as np
import awkward as ak 
import uproot as ur 

# ----------------------------------------------------------------------
# 1. Read the original file
# ----------------------------------------------------------------------
file = ur.open(r"data/last_Hijing_1M.root")

startt = 0
endd = -1 
tree = file["tree;1"]  # I have no clue why its jet_tree;1;1

phi = tree['phi'].array(entry_start=startt, entry_stop=endd)
pt = tree['pt'].array(entry_start=startt, entry_stop=endd)
eta = tree['eta'].array(entry_start=startt, entry_stop=endd)
weightss = tree['weight'].array(entry_start=startt, entry_stop=endd)

# ----------------------------------------------------------------------
# 2. Exact-inversion sampler for f(phi) = (1 + 2*v*cos(2phi)) / (2*pi)
#    v=0 gives a perfectly isotropic (uniform) distribution.
# ----------------------------------------------------------------------
def sample_phi_exact(n, v):
    """
    Draw n samples from
        f(phi) = (1 + 2*v*cos(2phi))/(2*pi)
    using inversion of the exact CDF with Newton iterations.
    v = 0 -> isotropic (uniform) phi.
    """
    out = np.empty(n)
    for i in range(n):
        u = np.random.random()
        phi_ = 2 * np.pi * u - np.pi          # initial guess: uniform
        for _ in range(4):                     # Newton iterations
            F = (phi_ + np.pi + v * np.sin(2 * phi_)) / (2 * np.pi) - u
            dF = (1 + 2 * v * np.cos(2 * phi_)) / (2 * np.pi)
            phi_ -= F / dF
        out[i] = phi_
    return out


# ----------------------------------------------------------------------
# 3. Keep the original phi (and eta/pt/weight) exactly as read from the
#    file -- no resampling of the real particles.
# ----------------------------------------------------------------------
lengths = np.fromiter((len(x) for x in phi), dtype=np.int64)
n_events = len(lengths)
N = lengths.sum()

phi = [np.array(x) for x in phi]      # unchanged, original phi values
eta = [np.array(x) for x in eta]
pt = [np.array(x) for x in pt]
weights = [np.array(x) for x in weightss] 

# ----------------------------------------------------------------------
# 4. Generate the 50 extra particles per event 
#    - phi:    isotropic (v = 0)
#    - eta/pt: bootstrap-sampled from the POOLED distribution across
#              every event (not just that event's own eta/pt)
#    - weight: exactly 1.0
# ----------------------------------------------------------------------
N_NEW_PER_EVENT = 50
total_new = n_events * N_NEW_PER_EVENT

# Pooled eta / pt values across ALL events (the "total dist")
eta_pool = np.concatenate(eta)
pt_pool = np.concatenate(pt) 
#weight_pool = np.concatenate(weights)

# Isotropic phi for the new particles (vectorized generation, then reshape)
new_phi_flat = sample_phi_exact(total_new, 0.05)
new_phi = new_phi_flat.reshape(n_events, N_NEW_PER_EVENT)

# Bootstrap sample eta/pt from the pooled distributions
new_eta = np.random.choice(eta_pool, size=total_new, replace=True).reshape(n_events, N_NEW_PER_EVENT)
new_pt = np.random.choice(pt_pool, size=total_new, replace=True).reshape(n_events, N_NEW_PER_EVENT)
#new_weights = np.random.choice(weight_pool, size=total_new, replace=True).reshape(n_events, N_NEW_PER_EVENT)

# ----------------------------------------------------------------------
# 5. Append the new particles to the end of each event's existing arrays
# ----------------------------------------------------------------------
final_phi = [np.concatenate([phi[i], new_phi[i]]) for i in range(n_events)]
final_eta = [np.concatenate([eta[i], new_eta[i]]) for i in range(n_events)]
final_pt = [np.concatenate([pt[i], new_pt[i]]) for i in range(n_events)]
#final_weights = [np.concatenate([weights[i], new_weights[i]]) for i in range(n_events)]

# ----------------------------------------------------------------------
# 6. Write the final (original + 50 new per event) arrays to a new ROOT file
# ----------------------------------------------------------------------
out_phi = ak.Array(final_phi)
out_eta = ak.Array(final_eta) 
out_pt = ak.Array(final_pt)
#out_weight = ak.Array(final_weights)

with ur.recreate("data/p_PB_50isomerge_v205.root") as fout:
    fout["tree"] = {
        "phi": out_phi,
        "eta": out_eta,
        "pt": out_pt,
        "weight": weightss,  
    }

print("Wrote data/p_PB_50isomerge_v205.root") 
print(f"Events: {n_events}, original particles: {N}, new particles per event: {N_NEW_PER_EVENT}")    

Wrote data/p_PB_50isomerge_v205.root
Events: 999999, original particles: 81741337, new particles per event: 50


In [ ]:
# adding isometric data shitt to p-p, set amount of points

import numpy as np
import awkward as ak 
import uproot as ur  

# ----------------------------------------------------------------------
# 1. Read the original file
# ----------------------------------------------------------------------
file = ur.open(r"data/pp_mb_fixed.root")

startt = 0
#endd = 8000000
endd = 8000000
tree = file["tree;1"]  # I have no clue why its jet_tree;1;1

phi = tree['phi'].array(entry_start=startt, entry_stop=endd)
pt = tree['pt'].array(entry_start=startt, entry_stop=endd)
eta = tree['eta'].array(entry_start=startt, entry_stop=endd)
weightss = tree['weight'].array(entry_start=startt, entry_stop=endd)
del file
del tree
# ----------------------------------------------------------------------
# 2. Exact-inversion sampler for f(phi) = (1 + 2*v*cos(2phi)) / (2*pi)
#    v=0 gives a perfectly isotropic (uniform) distribution.
# ----------------------------------------------------------------------
def sample_phi_exact(n, v):
    """
    Draw n samples from
        f(phi) = (1 + 2*v*cos(2phi))/(2*pi)
    using inversion of the exact CDF with Newton iterations.
    v = 0 -> isotropic (uniform) phi.
    """
    out = np.empty(n)
    for i in range(n):
        u = np.random.random()
        phi_ = 2 * np.pi * u - np.pi          # initial guess: uniform
        for _ in range(4):                     # Newton iterations
            F = (phi_ + np.pi + v * np.sin(2 * phi_)) / (2 * np.pi) - u
            dF = (1 + 2 * v * np.cos(2 * phi_)) / (2 * np.pi)
            phi_ -= F / dF
        out[i] = phi_
    return out


# ----------------------------------------------------------------------
# 3. Keep the original phi (and eta/pt/weight) exactly as read from the
#    file -- no resampling of the real particles.
# ----------------------------------------------------------------------
lengths = np.fromiter((len(x) for x in phi), dtype=np.int64)
mult_mask = (lengths>=10)

phi = phi[mult_mask]
eta = eta[mult_mask]
pt = pt[mult_mask]
weightss = weightss[mult_mask]
lengths = lengths[mult_mask]

n_events = len(lengths)
N = lengths.sum()

phi = [np.array(x) for x in phi]      # unchanged, original phi values
eta = [np.array(x) for x in eta]
pt = [np.array(x) for x in pt]
weights = [np.array(x) for x in weightss] 

# ----------------------------------------------------------------------
# 4. Generate the 50 extra particles per event
#    - phi:    isotropic (v = 0)
#    - eta/pt: bootstrap-sampled from the POOLED distribution across
#              every event (not just that event's own eta/pt)
#    - weight: exactly 1.0
# ----------------------------------------------------------------------
N_NEW_PER_EVENT = 110
total_new = n_events * N_NEW_PER_EVENT

# Pooled eta / pt values across ALL events (the "total dist")
eta_pool = np.concatenate(eta)
pt_pool = np.concatenate(pt) 
#weight_pool = np.concatenate(weights)

# Isotropic phi for the new particles (vectorized generation, then reshape)
new_phi_flat = sample_phi_exact(total_new, 0.05)
new_phi = new_phi_flat.reshape(n_events, N_NEW_PER_EVENT)

# Bootstrap sample eta/pt from the pooled distributions
new_eta = np.random.choice(eta_pool, size=total_new, replace=True).reshape(n_events, N_NEW_PER_EVENT)
new_pt = np.random.choice(pt_pool, size=total_new, replace=True).reshape(n_events, N_NEW_PER_EVENT)
#new_weights = np.random.choice(weight_pool, size=total_new, replace=True).reshape(n_events, N_NEW_PER_EVENT)

# ----------------------------------------------------------------------
# 5. Append the new particles to the end of each event's existing arrays
# ----------------------------------------------------------------------
final_phi = [np.concatenate([phi[i], new_phi[i]]) for i in range(n_events)]
final_eta = [np.concatenate([eta[i], new_eta[i]]) for i in range(n_events)]
final_pt = [np.concatenate([pt[i], new_pt[i]]) for i in range(n_events)]
#final_weights = [np.concatenate([weights[i], new_weights[i]]) for i in range(n_events)]

# ----------------------------------------------------------------------
# 6. Write the final (original + 50 new per event) arrays to a new ROOT file
# ----------------------------------------------------------------------
out_phi = ak.Array(final_phi)
out_eta = ak.Array(final_eta) 
out_pt = ak.Array(final_pt)
#out_weight = ak.Array(final_weights) 

with ur.recreate("data/pp_isomerge_all.root") as fout:
    fout["tree"] = {
        "phi": out_phi,
        "eta": out_eta,
        "pt": out_pt,
        "weight": weightss,   
    }

print("Wrote data/pp_isomerge.root") 
print(f"Events: {n_events}, original particles: {N}, new particles per event: {N_NEW_PER_EVENT}")    

In [5]:
1+1

2

In [1]:
#above but with AI
# adding isotropic particles to p-p, memory-optimized (chunked/streaming) version
#
# Key memory changes vs. the original:
#   1. Never loads the whole 8M-event dataset into memory at once. Both the
#      "build the eta/pt pool" step and the "generate + write" step stream
#      the tree in chunks via tree.iterate(...).
#   2. Never converts branches into Python lists of numpy arrays (that
#      per-event object overhead is enormous at 8M events) -- everything
#      stays as awkward arrays / plain numpy arrays.
#   3. sample_phi_exact is fully vectorized with numpy (no per-sample
#      Python-level loop), so it's cheap to call it once per chunk.
#   4. Generated arrays use float32 (matching typical ntuple branch dtype)
#      instead of float64, halving the memory of the new-particle arrays,
#      which are the single biggest memory consumer (n_events * 110).
#   5. Per-chunk temporaries are deleted and gc.collect()'d so peak memory
#      stays roughly constant regardless of total dataset size.

import gc
import numpy as np
import awkward as ak
import uproot as ur

IN_PATH = r"data/pp_mb_fixed.root"
OUT_PATH = r"data/pp_isomerge_all.root"

startt = 0
endd = -1
MIN_MULT = 10
N_NEW_PER_EVENT = 110

# Tune this based on available RAM. At CHUNK_SIZE events per chunk, the
# generated-particle arrays are ~CHUNK_SIZE * N_NEW_PER_EVENT floats each
# (phi/eta/pt), so e.g. 200_000 * 110 * 4 bytes (float32) ~= 88 MB per
# branch per chunk -- easily bounded regardless of how many total events
# (8M here) the file has.
CHUNK_SIZE = 200_000

rng = np.random.default_rng()

# ----------------------------------------------------------------------
# Vectorized exact-inversion sampler for f(phi) = (1 + 2*v*cos(2phi)) / (2*pi)
# v=0 gives a perfectly isotropic (uniform) distribution.
# Fully vectorized (no Python-level per-sample loop) for speed + to avoid
# per-iteration scalar/object overhead when called on large chunks.
# ----------------------------------------------------------------------
def sample_phi_exact_vec(n, v, dtype=np.float32):
    u = rng.random(n)
    phi_ = 2 * np.pi * u - np.pi          # initial guess: uniform
    for _ in range(4):                     # Newton iterations
        F = (phi_ + np.pi + v * np.sin(2 * phi_)) / (2 * np.pi) - u
        dF = (1 + 2 * v * np.cos(2 * phi_)) / (2 * np.pi)
        phi_ -= F / dF
    return phi_.astype(dtype, copy=False)


# ----------------------------------------------------------------------
# PASS 1: stream through the file just to build the pooled eta/pt
# distributions (only from events passing the multiplicity cut). Only
# reads the 'eta' and 'pt' branches -- 'phi' and 'weight' aren't needed
# here, which keeps this pass as cheap as possible.
# ----------------------------------------------------------------------
eta_pool_chunks = []
pt_pool_chunks = []
n_events_survivors = 0
N_total = 0 

with ur.open(IN_PATH) as file:
    tree = file["tree;1"]  # I have no clue why its jet_tree;1;1

    for chunk in tree.iterate(
        expressions=["eta", "pt"],
        entry_start=startt,
        entry_stop=endd,
        step_size=CHUNK_SIZE,
        library="ak",
    ):
        lengths_chunk = ak.num(chunk["eta"], axis=1)
        mask = lengths_chunk >= MIN_MULT

        eta_masked = chunk["eta"][mask]
        pt_masked = chunk["pt"][mask]

        eta_pool_chunks.append(ak.to_numpy(ak.flatten(eta_masked)).astype(np.float32, copy=False))
        pt_pool_chunks.append(ak.to_numpy(ak.flatten(pt_masked)).astype(np.float32, copy=False))

        n_events_survivors += int(ak.sum(mask))
        N_total += int(ak.sum(lengths_chunk[mask]))

        del chunk, lengths_chunk, mask, eta_masked, pt_masked
        gc.collect()

eta_pool = np.concatenate(eta_pool_chunks)
pt_pool = np.concatenate(pt_pool_chunks)
del eta_pool_chunks, pt_pool_chunks
gc.collect()

print(f"Pass 1 done: kept {n_events_survivors} events with multiplicity >= {MIN_MULT}, "
      f"pooled from {N_total} original particles.")

# ----------------------------------------------------------------------
# PASS 2: stream through the file again, this time reading all 4
# branches, applying the same multiplicity cut, generating the new
# particles per surviving event, merging them onto the end of each
# event's original arrays, and writing the result straight to the
# output file chunk-by-chunk (no full-dataset array is ever held in
# memory).
# ----------------------------------------------------------------------
with ur.open(IN_PATH) as file, ur.recreate(OUT_PATH) as fout:
    tree = file["tree;1"]
    first_chunk = True

    for chunk in tree.iterate(
        expressions=["phi", "pt", "eta", "weight"],
        entry_start=startt,
        entry_stop=endd,
        step_size=CHUNK_SIZE,
        library="ak",
    ):
        lengths_chunk = ak.num(chunk["phi"], axis=1)
        mask = lengths_chunk >= MIN_MULT

        phi_masked = chunk["phi"][mask]
        eta_masked = chunk["eta"][mask]
        pt_masked = chunk["pt"][mask]
        weight_masked = chunk["weight"][mask]

        n_chunk_survivors = int(ak.sum(mask))
        total_new_chunk = n_chunk_survivors * N_NEW_PER_EVENT

        # Isotropic-ish phi for the new particles (vectorized, float32)
        new_phi_flat = sample_phi_exact_vec(total_new_chunk, 0.05, dtype=np.float32)
        new_phi_2d = new_phi_flat.reshape(n_chunk_survivors, N_NEW_PER_EVENT)

        # Bootstrap sample eta/pt from the globally-pooled distributions
        new_eta_2d = rng.choice(eta_pool, size=total_new_chunk, replace=True).reshape(
            n_chunk_survivors, N_NEW_PER_EVENT
        )
        new_pt_2d = rng.choice(pt_pool, size=total_new_chunk, replace=True).reshape(
            n_chunk_survivors, N_NEW_PER_EVENT
        )

        # Merge original (jagged) + new (regular) per event -> jagged
        final_phi_chunk = ak.concatenate([phi_masked, ak.Array(new_phi_2d)], axis=1)
        final_eta_chunk = ak.concatenate([eta_masked, ak.Array(new_eta_2d)], axis=1)
        final_pt_chunk = ak.concatenate([pt_masked, ak.Array(new_pt_2d)], axis=1)
        final_weight_chunk = weight_masked  # unchanged, not extended for new particles

        out_data = {
            "phi": final_phi_chunk,
            "eta": final_eta_chunk,
            "pt": final_pt_chunk,
            "weight": final_weight_chunk,
        }

        if first_chunk:
            fout["tree"] = out_data
            first_chunk = False
        else:
            fout["tree"].extend(out_data)

        del (chunk, lengths_chunk, mask, phi_masked, eta_masked, pt_masked, weight_masked,
            new_phi_flat, new_phi_2d, new_eta_2d, new_pt_2d,
            final_phi_chunk, final_eta_chunk, final_pt_chunk, final_weight_chunk, out_data)
        gc.collect()

print(f"Wrote {OUT_PATH}")
print(f"Events: {n_events_survivors}, original particles: {N_total}, "
      f"new particles per event: {N_NEW_PER_EVENT}")

Pass 1 done: kept 4448945 events with multiplicity >= 10, pooled from 136191599 original particles.
Wrote data/pp_isomerge_all.root
Events: 4448945, original particles: 136191599, new particles per event: 110


In [1]:
#this should be the making ht edata with vs varying based on pt
import gc
import numpy as np
import awkward as ak
import uproot as ur

IN_PATH = r"data/pp_mb_fixed.root"
OUT_PATH = r"data/pp_isomerge_v209.root"
HIST_PATH = r"data/p_Pb_eta_pt_histograms.npz" 

startt = 0
endd = 4000000
MIN_MULT = 10
N_NEW_PER_EVENT = 110

# Tune this based on available RAM. At CHUNK_SIZE events per chunk, the
# generated-particle arrays are ~CHUNK_SIZE * N_NEW_PER_EVENT floats each
# (phi/eta/pt), so e.g. 200_000 * 110 * 4 bytes (float32) ~= 88 MB per
# branch per chunk -- easily bounded regardless of how many total events
# the file has.
CHUNK_SIZE = 200_000

rng = np.random.default_rng()

# ----------------------------------------------------------------------
# pt -> v2 mapping for the injected particles.
#   pt in [0, 2)   -> v2 = 0.05
#   pt in [2, 4)   -> v2 = 0.8
#   pt in [4, 6)   -> v2 = 0.1
#   pt >= 6        -> v2 = 0.8
# ----------------------------------------------------------------------
PT_BIN_EDGES = np.array([2.0, 4.0, 6.0])   # internal edges only
V2_BY_PT_BIN = np.array([0.05, 0.8, 0.1, 0.8])

def v2_from_pt(pt):
    """Vectorized lookup: maps an array of pt values to their v2 (elliptic
    flow) parameter according to PT_BIN_EDGES / V2_BY_PT_BIN above."""
    idx = np.digitize(pt, PT_BIN_EDGES)  # 0,1,2,3 for the 4 bins
    return V2_BY_PT_BIN[idx]


# ----------------------------------------------------------------------
# Vectorized exact-inversion sampler for f(phi) = (1 + 2*v*cos(2phi)) / (2*pi)
# v=0 gives a perfectly isotropic (uniform) distribution.
# v can now be a scalar OR an array the same shape as n (one v2 per
# particle) -- the Newton iteration below is fully elementwise either way.
# ----------------------------------------------------------------------
def sample_phi_exact_vec(n, v, dtype=np.float32):
    u = rng.random(n)
    phi_ = 2 * np.pi * u - np.pi          # initial guess: uniform
    for _ in range(4):                     # Newton iterations
        F = (phi_ + np.pi + v * np.sin(2 * phi_)) / (2 * np.pi) - u
        dF = (1 + 2 * v * np.cos(2 * phi_)) / (2 * np.pi)
        phi_ -= F / dF
    return phi_.astype(dtype, copy=False)


# ----------------------------------------------------------------------
# Histogram-based sampler: pick a bin according to its probability
# (counts/sum(counts)), then draw uniformly within that bin's edges.
# Vectorized over n at once.
# ----------------------------------------------------------------------
def sample_from_histogram(counts, edges, n, dtype=np.float32):
    probs = counts / counts.sum()
    bin_idx = rng.choice(len(counts), size=n, p=probs)
    left = edges[bin_idx]
    right = edges[bin_idx + 1]
    return rng.uniform(left, right).astype(dtype, copy=False)


# ----------------------------------------------------------------------
# Load the pre-saved eta/pt histograms once (small, constant memory --
# no need to build a pool from the dataset itself anymore).
# ----------------------------------------------------------------------
hist_data = np.load(HIST_PATH)
eta_counts, eta_edges = hist_data["eta_counts"], hist_data["eta_edges"]
pt_counts, pt_edges = hist_data["pt_counts"], hist_data["pt_edges"]

n_events_survivors = 0 
N_total = 0 

# ----------------------------------------------------------------------
# SINGLE PASS: stream through the file, apply the multiplicity cut,
# generate the new particles per surviving event from the histograms
# above, merge them onto the end of each event's original arrays, and
# write the result straight to the output file chunk-by-chunk (no
# full-dataset array is ever held in memory).
# ----------------------------------------------------------------------
with ur.open(IN_PATH) as file, ur.recreate(OUT_PATH) as fout:
    tree = file["tree;1"]  # I have no clue why its jet_tree;1;1
    first_chunk = True

    for chunk in tree.iterate(
        expressions=["phi", "pt", "eta", "weight"],
        entry_start=startt,
        entry_stop=endd,
        step_size=CHUNK_SIZE,
        library="ak",
    ):
        lengths_chunk = ak.num(chunk["phi"], axis=1)
        mask = lengths_chunk >= MIN_MULT

        phi_masked = chunk["phi"][mask]
        eta_masked = chunk["eta"][mask]
        pt_masked = chunk["pt"][mask]
        weight_masked = chunk["weight"][mask]

        n_chunk_survivors = int(ak.sum(mask))
        total_new_chunk = n_chunk_survivors * N_NEW_PER_EVENT

        # --- Sample pt FIRST (flat), since phi now depends on it ---
        new_pt_flat = sample_from_histogram(pt_counts, pt_edges, total_new_chunk)

        # --- Map each new particle's pt to its v2, then sample phi ---
        new_v2_flat = v2_from_pt(new_pt_flat)
        new_phi_flat = sample_phi_exact_vec(total_new_chunk, new_v2_flat, dtype=np.float32)

        new_phi_2d = new_phi_flat.reshape(n_chunk_survivors, N_NEW_PER_EVENT)
        new_pt_2d = new_pt_flat.reshape(n_chunk_survivors, N_NEW_PER_EVENT)

        # eta sampling unchanged / independent of pt or phi
        new_eta_2d = sample_from_histogram(eta_counts, eta_edges, total_new_chunk).reshape(
            n_chunk_survivors, N_NEW_PER_EVENT
        )

        # Merge original (jagged) + new (regular) per event -> jagged
        final_phi_chunk = ak.concatenate([phi_masked, ak.Array(new_phi_2d)], axis=1)
        final_eta_chunk = ak.concatenate([eta_masked, ak.Array(new_eta_2d)], axis=1)
        final_pt_chunk = ak.concatenate([pt_masked, ak.Array(new_pt_2d)], axis=1)
        final_weight_chunk = weight_masked  # unchanged, not extended for new particles

        out_data = {
            "phi": final_phi_chunk,
            "eta": final_eta_chunk,
            "pt": final_pt_chunk,
            "weight": final_weight_chunk,
        }

        if first_chunk:
            fout["tree"] = out_data
            first_chunk = False
        else:
            fout["tree"].extend(out_data)

        n_events_survivors += n_chunk_survivors
        N_total += int(ak.sum(lengths_chunk[mask]))

        del (chunk, lengths_chunk, mask, phi_masked, eta_masked, pt_masked, weight_masked,
            new_pt_flat, new_v2_flat, new_phi_flat, new_phi_2d, new_eta_2d, new_pt_2d,
            final_phi_chunk, final_eta_chunk, final_pt_chunk, final_weight_chunk, out_data)
        gc.collect()

print(f"Wrote {OUT_PATH}")
print(f"Events: {n_events_survivors}, original particles: {N_total}, "
      f"new particles per event: {N_NEW_PER_EVENT}") 

In [2]:
1+1 

2

In [8]:
import numpy as np   
import pandas as pd      
import uproot as ur     

file = ur.open(r"data/merged_100_200.root")

startt = 0 
endd = 100000
tree = file["tree;1"]   #I have no clue why its jet_tree;1;1
phi  = tree['phi'].array(entry_start = startt, entry_stop = endd)  # Replace with actual tree name
pt = tree['pt'].array(entry_start = startt, entry_stop = endd)
eta = tree['eta'].array(entry_start = startt, entry_stop = endd)
weights = tree['weight'].array(entry_start = startt, entry_stop = endd) 
#pt_flat = np.ndarray.flatten(pt)

#this was fornudging v2 to teh v2 we want to see if cor4 is negative then, was from ai, but if it works it works

#import random.uniform as rnduin

from scipy.stats import rv_continuous

    
def sample_phi_exact(n, v):
    """
    Draw n samples from

        f(phi) = (1 + 2*v*cos(2phi))/(2*pi)

    using inversion of the exact CDF with Newton iterations.
    """
    out = np.empty(n)

    for i in range(n):

        # uniform random number
        u = np.random.random()

        # initial guess: uniform distribution
        phi = 2*np.pi*u - np.pi

        # Newton iterations
        for _ in range(4):
            F = (phi + np.pi + v*np.sin(2*phi))/(2*np.pi) - u
            dF = (1 + 2*v*np.cos(2*phi))/(2*np.pi)
            phi -= F/dF

        out[i] = phi

    return out
lengths = np.fromiter((len(x) for x in phi), dtype=np.int64)
N = lengths.sum()

# Compile on first call
sample_phi_exact(1, 0.05)

start = time.perf_counter()

phi_all = sample_phi_exact(N, 0.05) 

phi = np.split(phi_all, np.cumsum(lengths[:-1]))

#phi = [np.array(x) for x in phi]
eta = [np.array(x) for x in eta]
pt = [np.array(x) for x in pt]
weights = weights/np.sum(weights)
testing12 = [[x, z, w, y] for x, z, w, y in zip(phi, weights, pt, eta)]

In [11]:
pt_flat = pt.flatten()

AttributeError: 'list' object has no attribute 'flatten'

In [15]:

import numpy as np   
import pandas as pd      
import uproot as ur     

file = ur.open(r"data/merged_100_200.root") #, not events in mult range 60-120 is about 640000 
#file = ur.open(r"data/pbpb_mb_500k.root")  
#file = ur.open(r"data/p_pb_jet.root") 
#file = ur.open(r"data/2merged_2pt.root")  

# List all keys (e.g., trees, histograms)     
print(file.keys())    
  
# Access a TTree       
startt = 0 
endd = -1
tree = file["tree;1"]   #I have no clue why its jet_tree;1;1
phi  = tree['phi'].array(entry_start = startt, entry_stop = endd)  # Replace with actual tree name
pt = tree['pt'].array(entry_start = startt, entry_stop = endd)
eta = tree['eta'].array(entry_start = startt, entry_stop = endd)
weights = tree['weight'].array(entry_start = startt, entry_stop = endd) 

#this was fornudging v2 to teh v2 we want to see if cor4 is negative then, was from ai, but if it works it works

#import random.uniform as rnduin

from scipy.stats import rv_continuous

    
class MyPDF(rv_continuous):
    def _pdf(self, x, value):
        return 1/2/np.pi*(1+2*value*np.cos(2*x)) 
dist = MyPDF(a=-np.pi, b=np.pi)
# tree = file["jet_tree;1"]   #I have no clue why its jet_tree;1;1
# phi  = tree['vtrackphi'].array(entry_start = startt, entry_stop = endd)  # Replace with actual tree name
# weights = tree['weight'].array(entry_start = startt, entry_stop = endd)
# pt = tree['vtrackpt'].array(entry_start = startt, entry_stop = endd)


#eta = tree['eta'].array(entry_start = startt, entry_stop = endd)
#phi = [np.array(nudge_v2(x, .01)) for x in phi]
#phi = dist.rvs(value=.05, size=len(phi))
phi = [np.array(x) for x in phi]
#phi = [dist.rvs(value=.05, size=len(x)) for x in phi]
eta = [np.array(x) for x in eta]
pt = [np.array(x) for x in pt]
weights = weights/np.sum(weights)
testing12 = [[x, z, w, y] for x, z, w, y in zip(phi, weights, pt, eta)]



['tree;1']


In [17]:
lengths = np.array([np.sum(a[2]>.4) for a in testing12])
print(np.min(lengths)) 
print(np.max(lengths)) 
print(len( lengths))

100
200
2547311


In [18]:
2547311/(15000000)

0.16982073333333333

In [8]:
print(uproot.__version__)

5.7.5


In [2]:
pip install awkward

Note: you may need to restart the kernel to use updated packages.
